# SITKit Quickstart / 快速入门

Use the bundled baseline to analyze one sample with a local **Qwen3-8B base model**.  
使用随 wheel 提供的 baseline，对本地 **Qwen3-8B 基础模型**进行单样本交互分析。

> Please open this Notebook from the SITKit repository root. / 请从 SITKit 根目录打开本 Notebook。

## 1. Import SITKit / 导入 SITKit

Install the official wheel before running this Notebook.  
运行前，请先按照 README 安装官方 wheel。

In [ ]:
# Run once if SITKit is not installed in this Jupyter kernel:
# %pip install ./sitkit-0.1.0-*.whl

from pathlib import Path
import json

from IPython.display import Image, display
from sitkit import SurrogateInteractionExplainer

## 2. Set three paths / 设置三个路径

Normally, only `MODEL_PATH` needs to be changed.  
通常只需要根据实际位置修改 `MODEL_PATH`。

In [ ]:
MODEL_PATH = Path("./models/Qwen3-8B")
DATASET_PATH = Path("examples/datasets/economics.json")
OUTPUT_DIR = Path("runs/notebook-quickstart")

## 3. Load one sample / 读取一个样本

The example uses the first sample in `economics.json`.  
示例使用 `economics.json` 中的第一个样本。

In [ ]:
samples = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
sample = samples[0]
sample_id = "economics-0001"

print("Text:", sample["text"])
print("Players:", [player["text"] for player in sample["players"]])

## 4. Run the analysis / 运行分析

SITKit checks the model and selects the matching official baseline automatically.  
SITKit 会自动校验模型并选择匹配的官方 baseline。

In [ ]:
explainer = SurrogateInteractionExplainer(MODEL_PATH)

result = explainer.explain(
    sample["text"],
    players=sample["players"],
    sample_id=sample_id,
    target_text=sample.get("target_text"),
    output_dir=OUTPUT_DIR,
)

payload = result.to_dict()
print("Analysis completed.")

## 5. View the strongest interactions / 查看主要交互

In [ ]:
model_result = payload["models"]["target_model"]

print("Positive TOP 5")
for row in model_result["positive_top"]:
    print(row["type"], row["player_ids"], row["value"])

print("\nNegative TOP 5")
for row in model_result["negative_top"]:
    print(row["type"], row["player_ids"], row["value"])

## 6. View the report / 查看报告

In [ ]:
report_dir = OUTPUT_DIR / sample_id / "report" / "target_model"

print((report_dir / "inference.txt").read_text(encoding="utf-8"))
print("Interaction table:", report_dir / "salient_interaction.csv")

for filename in (
    "optimization_loss.png",
    "signed_strength_by_order.png",
    "and_or_tree.png",
    "sparsity.png",
):
    display(Image(filename=str(report_dir / filename)))

## Optional: compare two models / 可选：比较两个模型

Add `reference_model` only when a second supported base model is needed.  
只有需要双模型比较时，才添加 `reference_model`。

In [ ]:
# reference_model = Path("./models/Meta-Llama-3-8B")
# explainer = SurrogateInteractionExplainer(
#     MODEL_PATH,
#     reference_model=reference_model,
# )